In [1]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import datetime

# Cấu hình giao diện đồ thị
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [2]:
# path="/home/slow_data/Air_Quality/filtered_envisoft_air_quality_weather_data.csv"
path="/home/slow_data/Air_Quality/IQAir_air_quality.csv"

In [3]:
df_aqi = pd.read_csv(path)
df_aqi.head()

,timestamp,station_name,longitude,latitude,aqi,WHO_exposure,PM2.5 (µg/m³),PM10 (µg/m³),O3 (µg/m³),NO2 (µg/m³),SO2 (µg/m³),CO (µg/m³),condition,temperature (°),humidity (%),pressure,wind_speed (km/h),wind_direction
0,2025-04-17 01:00:00,Hà Nội: Đại Học Bách Khoa cổng Parabol đường ...,105.8418,21.005200,167,15.6,78.1,215.0,62.8,9.6,8.2,NaN,Nhiều mây,23,86,1007,13.2,135
1,2025-04-17 01:00:00,Hà Nội: Công viên hồ điều hòa Nhân Chính Khuấ...,105.7947,21.003100,154,12.1,60.3,204.5,10.2,2.5,6.5,2.0,Nhiều mây,23,86,1007,13.4,136
2,2025-04-17 01:00:00,Minh Khai - Bắc Từ Liêm,105.7400,21.050000,132,5.2,26.2,218.7,21.0,NaN,0.1,1.4,Mưa,23,84,1007,12.3,132
3,2025-04-17 01:00:00,Vũng Tàu: Ngã tư Giếng nước - Tp.Vũng Tàu (KK),107.0844,10.367976,83,5.2,26.2,66.6,73.5,3.9,6.2,0.1,Nhiều mây,27,83,1010,19.0,103
4,2025-04-17 01:00:00,Hải Dương: UBND TP. Hải Dương - 106 Đường Trần...,106.3357,20.938100,144,10.6,53.0,147.4,48.2,1.0,1.3,2.2,Nhiều mây,21,88,1007,11.7,119


In [4]:
# QB_filter = df_aqi[df_aqi['Name'] == 'Quảng Bình: KKT Hòn La (KK)']
# QB_filter.head()

# Extract AOD for stations

In [5]:
import rasterio
import glob
from  pathlib import Path

In [6]:
# unique_stations = df_aqi['Name'].unique()
# station_df = df_aqi.groupby('Name')[['ID','Latitude', 'Longitude']].first().reset_index()

unique_stations = df_aqi['station_name'].unique()
station_df = df_aqi.groupby('station_name')[['latitude', 'longitude']].first().reset_index()

In [7]:
station_df

,station_name,latitude,longitude
0,"Công viên hồ điều hòa Nhân Chính, Khuất Duy Tiến",21.003100,105.794700
1,FPT,10.841600,106.809100
2,HCM - FPT Thuduc,10.841600,106.809100
3,Hà Nội: Chi cục BVMT (KK),21.015250,105.800130
4,Hà Nội: Công viên hồ điều hòa Nhân Chính Khuấ...,21.003100,105.794700
5,Hà Nội: TT giao lưu văn hóa phố cổ - Hoàn Kiếm...,21.035584,105.852771
6,Hà Nội: Đại Học Bách Khoa cổng Parabol đường ...,21.005200,105.841800
7,Hải Dương: UBND TP. Hải Dương - 106 Đường Trần...,20.938100,106.335700
8,IQAir Ha Noi,21.067900,105.826200
9,IQAir Vietnam - Saigon Pearl,10.790500,106.718700


In [8]:
aod_dir = "/home/slow_data/Air_Quality/AOD/L3_AOD"
OUTPUT_DIR = "/home/slow_data/Air_Quality/AOD/station_aod/L3_IQAir_stations"
aod_path = Path(aod_dir)

In [9]:
import os
import glob
import rasterio
import pandas as pd
import numpy as np

# Setup output directory to keep things organized
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Define the pattern
pattern = os.path.join(aod_dir, r'2025??/*/aod_vietnam_H??_*_1HARP031_FLDK.*.tif')
print("Search pattern:", pattern)
files = glob.glob(pattern)
files.sort() # Good practice to process in order
print(f"Found {len(files)} files.")

# Define the columns we want in the final CSVs
band_names = ['AOT_Merged', 'AOT_Pure', 'AOT_Merged_uncertainty', 'AOT_Pure_uncertainty', 'AE_Merged', 'AE_Pure', 'QA_flag_Merged', 'QA_flag_Pure', 'AOT_L2_Mean', 'AOT_L2_SDV', 'AOT_L2_Num', 'AE_L2_Mean', 'AE_L2_SDV', 'AE_L2_Num']
columns = ['timestamp'] + band_names

for aod_file in files:
    print("Processing file: ", aod_file)
    
    try:
        # 1. Parse Timestamp
        filename = os.path.basename(aod_file)
        parts = filename.split("_")
        timestamp = parts[3] + "_" + parts[4]

        # 2. Read Image Data
        with rasterio.open(aod_file) as src:
            # Read all required bands at once to keep memory handy
            # Bands are 1-indexed in rasterio
            bands_data = [src.read(i) for i in range(2, 16)] 

            
            # 3. Iterate through each station
            for _, row in station_df.iterrows():
                # station_id = str(row['ID'])
                # lon, lat = row["Longitude"], row["Latitude"]

                station_id = str(row['station_name'])
                lon, lat = row["longitude"], row["latitude"]
                
                # Define the output file path for this specific station
                station_csv_path = os.path.join(OUTPUT_DIR, f"{station_id}.csv")
                
                # timestamp = pd.to_datetime(timestamp, format='%Y%m%d_%H%M')
                extracted_values: dict[str, object] = {'timestamp': timestamp}
                
                try:
                    # Get pixel coordinates
                    rowcol = src.index(lon, lat)
                    
                    # Extract data for all bands
                    for i, name in enumerate(band_names):
                        # bands_data is 0-indexed list, so i=0 corresponds to Band 1
                        val = bands_data[i][rowcol[0], rowcol[1]]
                        extracted_values[name] = val
                        
                except Exception as e:
                    # If coordinate is out of bounds or error occurs, ensure band keys remain NaN
                    for name in band_names:
                        extracted_values[name] = np.nan

                # 4. Save to CSV (Append Mode)
                new_df = pd.DataFrame([extracted_values])
                
                # Check if file exists to determine if we need to write the header
                file_exists = os.path.isfile(station_csv_path)
                
                # mode='a' appends to the file instead of overwriting
                new_df.to_csv(station_csv_path, mode='a', header=not file_exists, index=False)

        print(f"✅ Processed timestamp {timestamp}")

    except Exception as e:
        print(f"❌ Error processing file {aod_file}: {e}")

print("All processing complete.")

Search pattern: /home/slow_data/Air_Quality/AOD/L3_AOD/2025??/*/aod_vietnam_H??_*_1HARP031_FLDK.*.tif
Found 8728 files.
Processing file:  /home/slow_data/Air_Quality/AOD/L3_AOD/202501/01/aod_vietnam_H09_20250101_0000_1HARP031_FLDK.02401_02401.tif
✅ Processed timestamp 20250101_0000
Processing file:  /home/slow_data/Air_Quality/AOD/L3_AOD/202501/01/aod_vietnam_H09_20250101_0100_1HARP031_FLDK.02401_02401.tif
✅ Processed timestamp 20250101_0100
Processing file:  /home/slow_data/Air_Quality/AOD/L3_AOD/202501/01/aod_vietnam_H09_20250101_0200_1HARP031_FLDK.02401_02401.tif
✅ Processed timestamp 20250101_0200
Processing file:  /home/slow_data/Air_Quality/AOD/L3_AOD/202501/01/aod_vietnam_H09_20250101_0300_1HARP031_FLDK.02401_02401.tif
✅ Processed timestamp 20250101_0300
Processing file:  /home/slow_data/Air_Quality/AOD/L3_AOD/202501/01/aod_vietnam_H09_20250101_0400_1HARP031_FLDK.02401_02401.tif
✅ Processed timestamp 20250101_0400
Processing file:  /home/slow_data/Air_Quality/AOD/L3_AOD/202501/0

In [20]:
dir = OUTPUT_DIR + "/FPT.csv"

df_aod = pd.read_csv(dir, parse_dates=[0])

In [21]:
df_aod.head()

,timestamp,AOT_Merged,AOT_Pure,AOT_Merged_uncertainty,AOT_Pure_uncertainty,AE_Merged,AE_Pure,QA_flag_Merged,QA_flag_Pure,AOT_L2_Mean,AOT_L2_SDV,AOT_L2_Num,AE_L2_Mean,AE_L2_SDV,AE_L2_Num
0,2025-01-01 07:00:00,NaN,NaN,NaN,NaN,NaN,NaN,1533.0,1533.0,NaN,NaN,0.0,NaN,NaN,0.0
1,2025-01-01 08:00:00,NaN,NaN,NaN,NaN,NaN,NaN,509.0,509.0,NaN,NaN,0.0,NaN,NaN,0.0
2,2025-01-01 09:00:00,NaN,NaN,NaN,NaN,NaN,NaN,509.0,509.0,NaN,NaN,0.0,NaN,NaN,0.0
3,2025-01-01 10:00:00,NaN,NaN,NaN,NaN,NaN,NaN,509.0,509.0,NaN,NaN,0.0,NaN,NaN,0.0
4,2025-01-01 11:00:00,NaN,NaN,NaN,NaN,NaN,NaN,509.0,509.0,NaN,NaN,0.0,NaN,NaN,0.0


In [18]:
df_aod.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8728 entries, 0 to 8727
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   timestamp               8728 non-null   datetime64[ns]
 1   AOT_Merged              34 non-null     float64       
 2   AOT_Pure                15 non-null     float64       
 3   AOT_Merged_uncertainty  11 non-null     float64       
 4   AOT_Pure_uncertainty    15 non-null     float64       
 5   AE_Merged               2 non-null      float64       
 6   AE_Pure                 6 non-null      float64       
 7   QA_flag_Merged          8728 non-null   float64       
 8   QA_flag_Pure            8728 non-null   float64       
 9   AOT_L2_Mean             1050 non-null   float64       
 10  AOT_L2_SDV              1050 non-null   float64       
 11  AOT_L2_Num              8728 non-null   float64       
 12  AE_L2_Mean              1050 non-null   float64 

In [19]:
import pandas as pd
import glob
import os

# Get all CSV files
csv_files = glob.glob(os.path.join(OUTPUT_DIR, "*.csv"))
print(f"Found {len(csv_files)} files to update.")

for file_path in csv_files:
    try:
        # 1. Read the CSV
        df = pd.read_csv(file_path)

        df['timestamp'] = pd.to_datetime(df['timestamp'], format='%Y%m%d_%H%M')
        
        # 3. Add 7 hours to convert UTC -> GMT+7
        df['timestamp'] = df['timestamp'] + pd.Timedelta(hours=7)

        df['timestamp'] = df['timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')
        
        # 5. Overwrite the file
        df.to_csv(file_path, index=False)
        
        print(f"Updated {os.path.basename(file_path)}")
        
    except Exception as e:
        print(f"❌ Error updating {file_path}: {e}")

print("✅ Timezone conversion complete.")

Found 19 files to update.
❌ Error updating /home/slow_data/Air_Quality/AOD/station_aod/L3_IQAir_stations/Quảng Bình: Khu kinh tế Hòn La (KK).csv: time data "2025-01-01 07:00:00" doesn't match format "%Y%m%d_%H%M", at position 0. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.
❌ Error updating /home/slow_data/Air_Quality/AOD/station_aod/L3_IQAir_stations/Hà Nội: Chi cục BVMT (KK).csv: time data "2025-01-01 07:00:00" doesn't match format "%Y%m%d_%H%M", at position 0. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, 

In [22]:
import pandas as pd
import numpy as np
from pathlib import Path

# 1. Setup the directory path
# source_dir = Path(OUTPUT_DIR)
source_dir = Path("/home/slow_data/Air_Quality/AOD/station_aod/L3_envisoft_stations")

# Check if directory exists
if not source_dir.exists():
    print(f"Error: Directory '{source_dir}' not found.")
else:
    # 2. Iterate through all .csv files in the directory
    csv_files = list(source_dir.glob('*.csv'))
    
    print(f"Found {len(csv_files)} CSV files. Processing...")

    for file_path in csv_files:
        try:
            # Read the CSV
            # explicit conversion isn't strictly necessary here, 
            # but reading it normally lets us target specific values safely
            df = pd.read_csv(file_path)
            
            # 3. Replace -9999.0 (and integer -9999) with NaN
            # We use a list to catch both float and int representations if mixed
            df.replace([-9999.0, -9999], np.nan, inplace=True)
            
            # 4. Save the file back (Overwriting the original)
            # index=False ensures we don't add a new index column every time we run this
            df.to_csv(file_path, index=False)
            
            print(f"Converted: {file_path.name}")
            
        except Exception as e:
            print(f"Failed to process {file_path.name}: {e}")

    print("Done.")

Found 13 CSV files. Processing...
Converted: 31388883344354363840031242796.csv
Converted: 28560877461938780203765592307.csv
Converted: 31388839920718814259329251882.csv
Converted: 31390903576425084107499649578.csv
Converted: 31388851800421997746903202346.csv
Converted: 29213751141295132066317063859.csv
Converted: 31390916083317566102523755051.csv
Converted: 31387251434693138681789561386.csv
Converted: 28505268571336961948594948504.csv
Converted: 31390932574706768021562473002.csv
Converted: 29195707587706641566224751462.csv
Converted: 31390908889087377344742439468.csv
Converted: 31390912357075263208060500522.csv
Done.


In [23]:
df_aod_2 = pd.read_csv("/home/slow_data/Air_Quality/AOD/station_aod/L3_envisoft_stations/28505268571336961948594948504.csv", parse_dates=[0])
df_aod_2.head()

,timestamp,AOT_Merged,AOT_Pure,AOT_Merged_uncertainty,AOT_Pure_uncertainty,AE_Merged,AE_Pure,QA_flag_Merged,QA_flag_Pure,AOT_L2_Mean,AOT_L2_SDV,AOT_L2_Num,AE_L2_Mean,AE_L2_SDV,AE_L2_Num
0,2025-01-01 07:00:00,NaN,NaN,NaN,NaN,NaN,NaN,1273.0,1273.0,NaN,NaN,0.0,NaN,NaN,0.0
1,2025-01-01 08:00:00,NaN,NaN,NaN,NaN,NaN,NaN,1273.0,1273.0,NaN,NaN,0.0,NaN,NaN,0.0
2,2025-01-01 09:00:00,NaN,NaN,NaN,NaN,NaN,NaN,505.0,505.0,NaN,NaN,0.0,NaN,NaN,0.0
3,2025-01-01 10:00:00,NaN,NaN,NaN,NaN,NaN,NaN,249.0,248.0,1.4006,0.1498,4.0,1.6741,0.3918,4.0
4,2025-01-01 11:00:00,NaN,NaN,NaN,NaN,NaN,NaN,509.0,509.0,1.2158,0.0944,2.0,0.9044,0.1097,2.0
